<center><u><H1>MLP for Regression</H1></u></center>

<img src="images/MLP_Regression.png" width="1000">

## Forward Propagation:

### <center> z_h = $XW^{(1)}$ </center>

### <center> a_h = $f$(z_h)</center> 

### <center> z_out = a_h $W^{(2)}$ </center>

### <center> $\hat{y}$ = $f$(z_out) </center>

## Cost Function:

## <center> C = $\frac{1}{2}$$\sum\limits_{i=1}^{n}$$(y_i-\hat{y_i})^2$ </center>

## Backpropagation:

### <center>$\delta_{out} = -(y - \hat{y}) * f'(z\_{out})$ </center>

### <center>$\delta_{hidden} = \delta_{out}(W_2^T) * f'(z\_{hidden})$ </center>

### <center>$\frac{dC}{dW^1} = grad\_w\_h = dot(X^T, \delta_{hidden})$ </center>

### <center>$\frac{dC}{dW^2} = grad\_w\_out = dot(a\_h^T, \delta_{out})$ </center>

### <center>$\frac{dC}{db_{hidden}} = grad\_b\_h = \sum(\delta_{hidden})$ </center>

### <center>$\frac{dC}{db_{out}} = grad\_b\_out = \sum(\delta_{out})$ </center>

### <center>$w\_h ~~ -= \lambda * grad\_w\_h$ </center>

### <center>$w\_out ~~ -= \lambda * grad\_w\_out$ </center>

### <center>$b\_h ~~ -= \lambda * grad\_b\_h$ </center>

### <center>$b\_out ~~ -= \lambda * grad\_b\_out$ </center>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## Plotting a Function:

In [ ]:
def data():
    X = np.sort(np.random.uniform(-4, 4,(200,1)),axis=0)
    y = np.exp(-0.05*X-0.5)*np.sin((0.03*X+0.7)*X)*4
    return X, y

In [ ]:
X, y = data()
fig, ax = plt.subplots(figsize=(8,6))
ax.plot(X, y,lw=2, color='green',label='Data')
ax.legend(loc='best')
ax.set_ylabel('Y values')
ax.set_xlabel('X values')

## Creating a class for a Neural Network

In [ ]:
class Neural_Network(object):
    def __init__(self):        
    #Hyperparameters
        self.inputLayerSize = 1
        self.outputLayerSize = 1
        self.hiddenLayerSize = 100
        self.random = np.random.RandomState(2018)
          
        self.W1 = self.random.normal(loc=0.0, scale=0.1,
                                    size=(self.inputLayerSize, self.hiddenLayerSize))
        self.W2 = self.random.normal(loc=0.0, scale=0.1,
                                    size=(self.hiddenLayerSize, self.outputLayerSize))
        
        self.b_h = np.zeros(self.hiddenLayerSize)
        self.b_out = np.zeros(self.outputLayerSize)
        
        self.epochs = 10000
        self.minibatches = 1
        self.learning_rate = 0.0001
        self.Lambda = 0.0001 #regularization parameter   

    ## Generating data:

    def testdata(self):
        X_test = np.sort(np.random.uniform(-4, 4,(200,1)),axis=0)
        y_test = np.exp(-0.05*X_test-0.5)*np.sin((0.03*X_test+0.7)*X_test)*4
        return X_test, y_test

    ## Activation Functions and Derivatives:

    def sigmoid(self, z):
        return 1/(1+np.exp(-z))

    def sigmoidPrime(self,z):
        return np.exp(-z)/((1+np.exp(-z))**2)
    
    def id_(self, z):
        return z
    
    def id_Prime(self, z):
        return 1

    ## Forward Propagation:

    def forward(self, X):
        self.z_h = np.dot(X, self.W1) + self.b_h
        self.a_h = self.sigmoid(self.z_h)
        self.z_out = np.dot(self.a_h, self.W2) + self.b_out
        self.a_out = self.id_(self.z_out) 
        return self.a_out

    ## Cost Function:

    def costFunction(self, X, y):
        yHat = self.forward(X)
        C = 0.5*np.sum((y-yHat)**2)/X.shape[0] + (0.5*self.Lambda)*(np.sum(self.W1**2)+np.sum(self.W2**2))
        return C

    def costFunctionPrime(self, X, y):            
        yHat = self.forward(X) 
        delta2 = -np.multiply((y-yHat), self.id_Prime(self.a_out)) 
        grad_b_out = np.sum(delta2, axis=0)
        dJdW2 = np.dot(self.a_h.T, delta2) + self.Lambda*self.W2
        delta1 = np.dot(delta2, self.W2.T)*self.sigmoidPrime(self.z_h)
        grad_b_h = np.sum(delta1, axis=0)
        dJdW1 = np.dot(X.T, delta1) + self.Lambda*self.W1
                           
        return dJdW1, dJdW2, grad_b_h, grad_b_out

    ## Creating the Neural Network model:

    def fit(self,X, y):
        cost_train = []
        cost_test = []
        X_train, y_train = X.copy(), y.copy()
        X_test, y_test = self.testdata()
        
        for i in range(self.epochs):
            mini = np.array_split(range(y_test.shape[0]), self.minibatches)
            for idx in mini:
                grad1, grad2, grad_b_h, grad_b_out = self.costFunctionPrime(X_train[idx], y_train[idx])
                self.W2 -= self.learning_rate * grad2
                self.W1 -= self.learning_rate * grad1
                self.b_h -= self.learning_rate * grad_b_h
                self.b_out -= self.learning_rate * grad_b_out
                
            costtrain = self.costFunction(X_train[idx], y_train[idx])
            cost_train.append(costtrain)
            costtest = self.costFunction(X_test[idx], y_test[idx])
            cost_test.append(costtest)
            
            if i % 1000 == 0:
                print('training-cost: %.2f' % costtrain)
                print('training-test: %.2f' % costtest) 
        
        y_hat = self.forward(X)
        
        return cost_train, cost_test, y_hat
    
    ## Plotting Cost Functions for training and testing:     
    def plot(self, cost_train, cost_test):
        fig, ax = plt.subplots(figsize=(8,6))
        ax.plot(np.arange(len(cost_train)), cost_train, lw=2, color='blue', label='CostTraining')
        ax.plot(np.arange(len(cost_test)), cost_test, lw=2, color='red', label='CostTest')
        ax.legend(loc='best')
        ax.set_ylabel('Cost')
        ax.set_xlabel('Cycles')
        return plt.show()

## Testing the model:

In [ ]:
NN = Neural_Network()

In [ ]:
cost_train, cost_test, y_hat = NN.fit(X,y)

In [ ]:
NN.plot(cost_train, cost_test)

In [ ]:
fig, ax = plt.subplots(figsize=(6,6))
ax.set_ylim(-5,5)
ax.set_xlim(-5,5)
ax.plot(X, y_hat, lw=2, color='red', label='Predictions')
ax.plot(X, y, lw=2, color='green', label='Data')